# Imbalanced TelecomTS Split (Scenario 2)

> **This notebook is the imbalanced-split twin of `../TelecomTS/SOTA_Baselines_TelecomTS.ipynb`.** The data layout under `data/` and `data/features_cache/` is identical in shape to the balanced version, but the underlying files are symlinks to the **1000-window TelecomTS split** (`../../split/TelecomTS_*.npz`) and the precomputed Chronos-2 residuals at the project root (`../../X_*_resid_full_norm.npy`).

| Split | Total | Normal | Anomaly | Positive rate |
|-------|------:|-------:|--------:|--------------:|
| Train |   640 |   608  |     32  |        ~5%   |
| Val   |   160 |   152  |      8  |        ~5%   |
| Test  |   200 |   190  |     10  |        ~5%   |

All BCE-trained methods auto-compute `pos_weight = N_neg / N_pos` from the training labels, so the only effective change vs the balanced notebook is the data path. Threshold-tunable methods continue to pick the best-F1 cutoff on the (also-imbalanced) validation split.

Results are written to `results/` in this folder.


# SOTA Anomaly-Detection Baselines on TelecomTS

**Goal.** Add 5 state-of-the-art multivariate time-series anomaly-detection baselines from top-tier ML conferences to the TelecomTS benchmark, evaluated on the **imbalanced 200-window TelecomTS test split** (190 normal / 10 anomaly; ~5% positive rate) — the same split used by the other baseline notebooks in this folder.

**Methods (paper / code).**

| # | Method | Venue | Family | Paper | Code |
|---|--------|-------|--------|-------|------|
| 1 | DCdetector | KDD 2023 | Dual-attention contrastive | https://arxiv.org/pdf/2306.10347 | https://github.com/DAMO-DI-ML/KDD2023-DCdetector |
| 2 | TimesNet   | ICLR 2023 | 2D-variation Inception | https://arxiv.org/pdf/2210.02186 | https://github.com/thuml/Time-Series-Library |
| 3 | ModernTCN  | ICLR 2024 (Spotlight) | Modern large-kernel TCN | https://openreview.net/pdf?id=vpJMJerXHU | https://github.com/luodhhh/ModernTCN |
| 4 | MEMTO      | NeurIPS 2023 | Memory-guided Transformer | https://papers.neurips.cc/paper_files/paper/2023/file/b4c898eb1fb556b8d871fbe9ead92256-Paper-Conference.pdf | https://github.com/gunny97/MEMTO |
| 5 | D3R        | NeurIPS 2023 | Dynamic decomposition + diffusion | https://openreview.net/pdf?id=aW5bSuduF1 | https://github.com/ForestsKing/D3R |

**Protocol.** Each method follows its original paper recipe:

* **Unsupervised training on normal-only windows** (`y == 0`) drawn from `train + val`.
* Per-step anomaly score (DCdetector: KL between dual attention maps; TimesNet/ModernTCN/D3R: per-step reconstruction MSE; MEMTO: bi-dim deviation = recon-MSE × gathering distance to nearest memory item).
* Window-level score = mean over time.
* Threshold selected by best F1 on the **validation split** windows; metrics reported on the held-out test split.
* No point-adjustment.

Input shape: `(N, 128, 16)` (128 timesteps × 16 KPIs).

> **Laptop / CPU mode.** Hyperparameters below are tuned to finish on a CPU-only laptop in ~minutes per method (smaller `d_model`, fewer epochs, single seed). To reproduce the strongest paper-faithful numbers later on a GPU, scale up: `SEEDS = [42, 123, 456, 789, 1337]`, restore `d_model` to the paper defaults (DCdetector 256, MEMTO 512, ModernTCN 64), and set epochs to {DCdetector 3, TimesNet 10, ModernTCN 10, MEMTO 3+3, D3R 10}.

## 1. Imports & device

In [ ]:
import os, sys, ssl, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings('ignore')

# Make the vendored sota_models package importable from the parent dir.
REPO_ROOT = Path('..').resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from sota_models import training as sota_train
from sota_models import common as sota_common

DEVICE = (
    'mps' if torch.backends.mps.is_available()
    else 'cuda' if torch.cuda.is_available()
    else 'cpu'
)
print(f'Device: {DEVICE} | torch {torch.__version__}')

OUT = Path('results'); OUT.mkdir(parents=True, exist_ok=True)

# HuggingFace SSL fix for macOS (matches Foundation_Models_TelecomTS.ipynb)
try:
    import certifi
    os.environ['SSL_CERT_FILE'] = certifi.where()
    os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()
    ssl._create_default_https_context = ssl._create_unverified_context
except Exception:
    pass

# Laptop/CPU default: 1 seed. For full 5-seed mean+/-std, set SEEDS = [42, 123, 456, 789, 1337].
SEEDS = [42]

## 2. Load the same `*_test.npz` used by every TelecomTS baseline

We additionally apply **per-channel z-normalization with training-set statistics**. The TelecomTS data is delivered raw (KPIs have heterogeneous scales: RSRP ≈ −80, throughput in MBps, BLER in [0, 1], …). Without this step, MEMTO/D3R have no internal normalization and their reconstruction loss is dominated by a few high-magnitude channels; AUROC collapses to ≈0.5 and the best-F1 threshold search degenerates to "predict every window as anomaly" (recall=1.0, precision=0.5, F1=0.667). SpotLight is already min-max normalized in `prepare_spotlight_dataset.py` and Production is z-normalized in its own data cell; this brings TelecomTS to the same footing.

In [ ]:
DATA_DIR = Path('data')
tr = np.load(DATA_DIR / 'TelecomTS_train.npz', allow_pickle=True)
vl = np.load(DATA_DIR / 'TelecomTS_val.npz',   allow_pickle=True)
te = np.load(DATA_DIR / 'TelecomTS_test.npz',  allow_pickle=True)

# Native shape from npz is (N, T, C); SOTA baselines expect that order.
X_train = tr['X'].astype(np.float32)
y_train = tr['y'].astype(np.int64)
X_val   = vl['X'].astype(np.float32)
y_val   = vl['y'].astype(np.int64)
X_test  = te['X'].astype(np.float32)
y_test  = te['y'].astype(np.int64)

# Robustness: NaN/Inf scrub then per-channel z-normalization with training-set statistics.
# Without this, raw KPI magnitudes (e.g. RSRP ~ -80, throughput in MBps, BLER in [0,1]) make
# reconstruction loss dominated by high-scale channels, AUROC collapses to ~0.5, and the
# best-F1 threshold search degenerates to "predict every window as anomaly". Production and
# SpotLight already pre-normalize their data; TelecomTS does not, so we apply it here.
X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
X_val   = np.nan_to_num(X_val,   nan=0.0, posinf=0.0, neginf=0.0)
X_test  = np.nan_to_num(X_test,  nan=0.0, posinf=0.0, neginf=0.0)

mu  = X_train.mean(axis=(0, 1), keepdims=True)            # per-channel mean over (N, T)
sig = X_train.std(axis=(0, 1),  keepdims=True) + 1e-8     # per-channel std over (N, T)
X_train = ((X_train - mu) / sig).astype(np.float32)
X_val   = ((X_val   - mu) / sig).astype(np.float32)
X_test  = ((X_test  - mu) / sig).astype(np.float32)

n_normal_train = int((y_train == 0).sum())
print(f'Train: {X_train.shape}  (anom ratio {y_train.mean():.3f}; normal-only used for training: {n_normal_train})')
print(f'Val:   {X_val.shape}    (anom ratio {y_val.mean():.3f})')
print(f'Test:  {X_test.shape}   (anom ratio {y_test.mean():.3f})')
print(f'After z-norm | train mean={X_train.mean():+.3f}, std={X_train.std():.3f} | '
      f'test mean={X_test.mean():+.3f}, std={X_test.std():.3f}')

WIN, K = X_train.shape[1], X_train.shape[2]
print(f'\nWindow length T={WIN}, KPI channels C={K}')

## 3. Multi-seed runner

For each method: train on normal-only windows in `train`, score on `val + test` together (so the model's weights are fixed when both splits are scored), pick a best-F1 threshold on validation, then report per-seed metrics and 5-seed mean ± std on test.

In [ ]:
def run_method(method: str, **kwargs) -> pd.DataFrame:
    rows = []
    for seed in SEEDS:
        t0 = time.time()
        print(f'\n=== {method} | seed={seed} ===')
        res = sota_train.evaluate_method(
            method, X_train, y_train, X_val, y_val, X_test, y_test,
            seed=seed, device=DEVICE, verbose=True, **kwargs,
        )
        elapsed = time.time() - t0
        print(f'  -> seed={seed} | F1={res["f1"]:.4f} AUROC={res["auroc"]:.4f} AP={res["ap"]:.4f} ({elapsed:.0f}s)')
        rows.append({
            'method': method, 'seed': seed,
            'precision': res['precision'], 'recall': res['recall'], 'f1': res['f1'],
            'auroc': res['auroc'], 'ap': res['ap'], 'threshold': res['threshold'],
            'elapsed_s': elapsed,
        })
    df = pd.DataFrame(rows)
    df.to_csv(OUT / f'sota_{method.lower()}_per_seed.csv', index=False)
    return df

## 4. DCdetector (KDD 2023)

Paper: Yang et al., *DCdetector: Dual Attention Contrastive Representation Learning for Time Series Anomaly Detection*, KDD 2023. https://arxiv.org/pdf/2306.10347

**Hyperparameters** (CPU-friendly): patch sizes ∈ {8, 16}, `d_model=64`, 2 layers, 1 head, batch=64, Adam lr=1e-4, 3 epochs. (Paper defaults: `d_model=256`, 3 layers, patch sizes ∈ {4, 8, 16}.)

In [ ]:
df_dcd = run_method('DCdetector',
                    patch_size=(8, 16),
                    d_model=64, e_layers=2, n_heads=1,
                    epochs=3, batch_size=64, lr=1e-4)
df_dcd

## 5. TimesNet (ICLR 2023)

Paper: Wu et al., *TimesNet: Temporal 2D-Variation Modeling for General Time Series Analysis*, ICLR 2023. https://arxiv.org/pdf/2210.02186

**Hyperparameters** (CPU-friendly): `e_layers=2`, `d_model=32`, `d_ff=32`, `top_k=2`, `num_kernels=4`, batch=128, Adam lr=1e-4, 5 epochs. (TSlib AD defaults: `d_model=64`, `top_k=3`, `num_kernels=6`, 10 epochs.)

In [ ]:
df_tn = run_method('TimesNet',
                   d_model=32, d_ff=32, e_layers=2, top_k=2, num_kernels=4,
                   epochs=5, batch_size=128, lr=1e-4)
df_tn

## 6. ModernTCN (ICLR 2024 Spotlight)

Paper: Luo & Wang, *ModernTCN: A Modern Pure Convolution Structure for General Time Series Analysis*, ICLR 2024. https://openreview.net/pdf?id=vpJMJerXHU

**Hyperparameters** (CPU-friendly): `large_size=31`, `small_size=5`, `dims=[32]`, `ffn_ratio=1`, `num_blocks=[1]`, patch_size=8, batch=128, Adam lr=1e-4, 5 epochs. (ModernTCN-detection defaults: `large_size=51`, `dims=[64]`, 10 epochs.)

In [ ]:
df_mt = run_method('ModernTCN',
                   patch_size=8,
                   large_size=31, small_size=5,
                   d_model=32, ffn_ratio=1, num_blocks=1,
                   epochs=5, batch_size=128, lr=1e-4)
df_mt

## 7. MEMTO (NeurIPS 2023)

Paper: Song et al., *MEMTO: Memory-guided Transformer for Multivariate Time Series Anomaly Detection*, NeurIPS 2023. https://papers.neurips.cc/paper_files/paper/2023/file/b4c898eb1fb556b8d871fbe9ead92256-Paper-Conference.pdf

**Hyperparameters** (CPU-friendly): 10-item memory, `d_model=64`, 4 heads, 2 encoder layers; loss = recon-MSE + 0.1 · gathering + 0.01 · entropy; batch=64, Adam lr=1e-4. Two-phase training (3 epochs random-init memory, k-means re-init, 3 more epochs). Test-time anomaly score = bi-dim deviation = recon-MSE × gathering distance. (Paper defaults: `d_model=512`, 3 encoder layers.)

In [ ]:
df_memto = run_method('MEMTO',
                      n_memory=10, d_model=64, n_heads=4, e_layers=2, d_ff=64,
                      lambda_gather=0.1, lambda_entropy=0.01,
                      epochs_phase1=3, epochs_phase2=3,
                      batch_size=64, lr=1e-4)
df_memto

## 8. D3R (NeurIPS 2023)

Paper: Wang et al., *Drift doesn't Matter: Dynamic Decomposition with Diffusion Reconstruction for Unstable Multivariate Time Series Anomaly Detection*, NeurIPS 2023. https://openreview.net/pdf?id=aW5bSuduF1

**Hyperparameters** (CPU-friendly): `model_dim=32`, `block_num=2`, 2 heads, diffusion `t=100`/`T=1000`, dynamic-decomposition delay `d=3`, time covariates = 4 sinusoidal positional features, batch=64, Adam lr=1e-4, 5 epochs. Test-time score = per-step diffusion reconstruction MSE. (Paper defaults: `model_dim=128`, `block_num=3`, 4 heads, `t=200`, 10 epochs.)

In [ ]:
df_d3r = run_method('D3R',
                    model_dim=32, ff_dim=32, atten_dim=8,
                    block_num=2, head_num=2, time_num=4,
                    t=100, d=3,
                    epochs=5, batch_size=64, lr=1e-4)
df_d3r

## 9. Aggregate summary table

Mean ± std over 5 seeds, plus a row appended to `results/benchmark_comparison.csv` so these methods show up alongside KAC and the existing baselines.

In [ ]:
all_dfs = {'DCdetector': df_dcd, 'TimesNet': df_tn, 'ModernTCN': df_mt, 'MEMTO': df_memto, 'D3R': df_d3r}

rows = []
for name, df in all_dfs.items():
    summary = sota_common.summarize_seeds(df.drop(columns=['method'], errors='ignore').to_dict('records'))
    rows.append({
        'method': name,
        'precision_mean': summary.get('precision_mean'), 'precision_std': summary.get('precision_std'),
        'recall_mean':    summary.get('recall_mean'),    'recall_std':    summary.get('recall_std'),
        'f1_mean':        summary.get('f1_mean'),        'f1_std':        summary.get('f1_std'),
        'auroc_mean':     summary.get('auroc_mean'),     'auroc_std':     summary.get('auroc_std'),
        'ap_mean':        summary.get('ap_mean'),        'ap_std':        summary.get('ap_std'),
    })
summary_df = pd.DataFrame(rows)
summary_df.to_csv(OUT / 'sota_baselines_results.csv', index=False)
print('Saved:', OUT / 'sota_baselines_results.csv')
summary_df

In [ ]:
# Append best-seed row per method into benchmark_comparison.csv (matching the existing schema).
bench_path = OUT / 'benchmark_comparison.csv'
if bench_path.exists():
    bench = pd.read_csv(bench_path)
    print(f'Existing rows in benchmark_comparison.csv: {len(bench)}')
else:
    bench = pd.DataFrame()
    print('No existing benchmark_comparison.csv -- will create one.')

new_rows = []
for name, df in all_dfs.items():
    best = df.loc[df['f1'].idxmax()]
    new_rows.append({
        'method': name,
        'precision': float(best['precision']),
        'recall':    float(best['recall']),
        'f1':        float(best['f1']),
        'auroc':     float(best['auroc']),
        'ap':        float(best['ap']),
    })
new_df = pd.DataFrame(new_rows)

# Drop any pre-existing rows for these methods so re-runs are idempotent.
if not bench.empty and 'method' in bench.columns:
    bench = bench[~bench['method'].isin(new_df['method'])].copy()
merged = pd.concat([bench, new_df], ignore_index=True)
merged.to_csv(bench_path, index=False)
print(f'Wrote {len(merged)} rows to {bench_path}')
merged.tail(8)